IoU_Dice_Analysis_Original_Augmented_Custom.ipynb

In [3]:
!pip install torch torchvision matplotlib

import torch
import torchvision
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Transformation for input images (resize + to tensor)
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

# Transformation for target masks
target_transform = transforms.Compose([
    transforms.Resize((128, 128), interpolation=Image.NEAREST),
    transforms.PILToTensor()
])

# Load the dataset
dataset = OxfordIIITPet(root='.', download=True, target_types='segmentation',
                        transform=transform, target_transform=target_transform)

# Split into train/test sets
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_ds, test_ds = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False)


100%|██████████| 792M/792M [00:26<00:00, 29.5MB/s]
100%|██████████| 19.2M/19.2M [00:01<00:00, 17.6MB/s]


In [5]:
from torchvision.models.segmentation import fcn_resnet50
import torch.nn.functional as F

# Load pretrained model
model = fcn_resnet50(weights="DEFAULT").eval()

def compute_iou_dice(pred, target):
    pred = pred.float()
    target = target.float()
    intersection = (pred * target).sum()
    union = (pred + target - pred * target).sum()
    dice = (2 * intersection) / (pred.sum() + target.sum() + 1e-7)
    iou = intersection / (union + 1e-7)
    return iou.item(), dice.item()

ious, dices = [], []

for imgs, masks in test_loader:
    with torch.no_grad():
        outputs = model(imgs)['out']   # shape: [B, C, H, W]
        preds = torch.argmax(outputs, dim=1)  # take class with max score

    # Masks in Oxford Pet: 1=pet, 2=border, 0=background
    masks = masks.squeeze(1)
    masks = torch.where(masks == 1, 1, 0)  # keep only pet as foreground

    preds = (preds == 15).float()  # COCO class 15 ≈ "cat/dog"-like (closest)

    for p, m in zip(preds, masks):
        i, d = compute_iou_dice(p, m)
        ious.append(i)
        dices.append(d)

print(f"Original Dataset → Mean IoU: {np.mean(ious):.4f}, Mean Dice: {np.mean(dices):.4f}")


Downloading: "https://download.pytorch.org/models/fcn_resnet50_coco-1167a1af.pth" to /root/.cache/torch/hub/checkpoints/fcn_resnet50_coco-1167a1af.pth


100%|██████████| 135M/135M [00:00<00:00, 151MB/s]


Original Dataset → Mean IoU: 0.0397, Mean Dice: 0.0519


In [7]:
from torchvision.datasets import OxfordIIITPet
from torchvision import transforms

# Define transforms (for augmentation or normalization)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Load augmented dataset with download=True
aug_dataset = OxfordIIITPet(
    root='.',                     # or a custom folder path like './data'
    target_types='segmentation',
    transform=transform,
    download=True                 # 🔹 This automatically downloads the dataset
)


In [8]:
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet
from torch.utils.data import DataLoader
from PIL import Image

# Augmentation transformations
aug_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor()
])

# Same for target masks (no color jitter, no flip)
target_transform = transforms.Compose([
    transforms.Resize((128, 128), interpolation=Image.NEAREST),
    transforms.PILToTensor()
])

# Load augmented dataset
aug_dataset = OxfordIIITPet(
    root='.',
    target_types='segmentation',
    transform=aug_transform,
    target_transform=target_transform
)

aug_loader = DataLoader(aug_dataset, batch_size=8, shuffle=False)

print("✅ Augmented dataset and loader are ready.")
print("Total augmented samples:", len(aug_loader.dataset))


✅ Augmented dataset and loader are ready.
Total augmented samples: 3680


In [ ]:
import torch

ious, dices = [], []

for imgs, masks in aug_loader:
    with torch.no_grad():
        outputs = model(imgs)['out']
        preds = torch.argmax(outputs, dim=1)

    masks = masks.squeeze(1)
    masks = torch.where(masks == 1, 1, 0)  # only pet area
    preds = (preds == 15).float()

    for p, m in zip(preds, masks):
        i, d = compute_iou_dice(p, m)
        ious.append(i)
        dices.append(d)

print(f"Augmented Dataset → Mean IoU: {np.mean(ious):.4f}, Mean Dice: {np.mean(dices):.4f}")


Augmented Dataset → Mean IoU: 0.0316, Mean Dice: 0.0460


In [13]:
import torch

def iou(pred, target, eps=1e-6):
    """Mean IoU for multi-class segmentation."""
    pred = pred.long()
    target = target.long()
    num_classes = int(torch.max(target)) + 1
    ious = []

    for cls in range(num_classes):
        pred_cls = (pred == cls)
        target_cls = (target == cls)

        intersection = (pred_cls & target_cls).sum().float()
        union = (pred_cls | target_cls).sum().float()

        if union == 0:
            continue
        ious.append(intersection / (union + eps))

    return torch.mean(torch.stack(ious)) if ious else torch.tensor(0.0)


def dice(pred, target, eps=1e-6):
    """Mean Dice for multi-class segmentation."""
    pred = pred.long()
    target = target.long()
    num_classes = int(torch.max(target)) + 1
    dices = []

    for cls in range(num_classes):
        pred_cls = (pred == cls)
        target_cls = (target == cls)

        intersection = (pred_cls & target_cls).sum().float()
        denom = pred_cls.sum().float() + target_cls.sum().float()

        if denom == 0:
            continue
        dices.append((2 * intersection + eps) / (denom + eps))

    return torch.mean(torch.stack(dices)) if dices else torch.tensor(0.0)


In [14]:
ious, dices = [], []

for imgs, masks in custom_loader:
    with torch.no_grad():
        outputs = model(imgs)['out']
        preds = torch.argmax(outputs, dim=1)
        ious.append(iou(preds, masks))
        dices.append(dice(preds, masks))

print(f"Custom Dataset → Mean IoU: {torch.mean(torch.tensor(ious)):.4f}, Mean Dice: {torch.mean(torch.tensor(dices)):.4f}")


Custom Dataset → Mean IoU: 0.8996, Mean Dice: 3.7733
